# 15.4 隐式反馈 / Implicit Feedback (BPR & weighted-MF)

**中文**：前三节都假设有"用户打了几星"的**显式评分**。但现实里 99% 的推荐系统**根本没有评分**——它们只有**点击、播放、购买、停留时长**这类**隐式反馈**。隐式反馈有三个让人头疼的特性，本节就是教你怎么对付它们。
**English**: The first three sections assumed **explicit ratings** ("user gave 4 stars"). But in reality 99% of recommenders **have no ratings at all** — only **clicks, plays, purchases, dwell time**: **implicit feedback**. Implicit feedback has three thorny properties, and this section is about handling them.

---

**中文**：隐式反馈的三大特性：
1. **只有正样本，没有负样本**：你点击了 = 正；你没点 = ？可能是不喜欢，**也可能只是没看到**。"没观测"≠"负"。
2. **没有偏好强度，只有"发生没发生"**：点了就是 1，没点就是 0（顶多有个次数/时长当置信度）。
3. **数据量巨大但噪声高**：一次误点、一个自动播放都算"正"。

**English**: Three defining properties of implicit feedback:
1. **Positives only, no explicit negatives**: a click = positive; a non-click = ? It could mean dislike, **or simply never seen**. "Unobserved" ≠ "negative."
2. **No preference strength, just occurrence**: clicked = 1, not = 0 (at most a count/dwell time as confidence).
3. **Huge volume, high noise**: a mis-click or an autoplay also counts as "positive."

**中文**：这意味着**不能**直接套用显式 MF（在 0/1 上做回归会把"没看到"硬学成"讨厌"）。两条主流路线：
**English**: This means you **cannot** naively reuse explicit MF (regressing on 0/1 would learn "never seen" as "hated"). Two mainstream routes:

- **BPR（贝叶斯个性化排序）**：把问题变成**两两排序**——"用户交互过的物品，应该排在没交互过的物品前面"。优化 pairwise ranking 而非逐点回归。
- **加权矩阵分解 / iALS（Hu-Koren-Volinsky 2008）**：保留所有 0/1，但给每个格子一个**置信度权重**——观测到的正样本权重高、未观测的权重低（但非零），用带权 ALS 求闭式解。

- **BPR (Bayesian Personalized Ranking)**: turn it into **pairwise ranking** — "an interacted item should rank above a non-interacted one." Optimize pairwise ranking, not pointwise regression.
- **Weighted MF / iALS (Hu-Koren-Volinsky 2008)**: keep all 0/1 but assign each cell a **confidence weight** — observed positives weigh high, unobserved low (but nonzero), solved in closed form by weighted ALS.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 工业必考）**
> **中文**：**隐式 vs 显式**：工业推荐几乎全是隐式。**负采样**是关键——没有真负样本，就从"未观测"里随机采样当负样本（BPR）。**BPR loss** = $-\log\sigma(\hat x_{ui}-\hat x_{uj})$，i 是正样本、j 是采样负样本。**iALS** 的核心技巧：置信度 $c_{ui}=1+\alpha r_{ui}$，并用 $Q^\top C^u Q = Q^\top Q + Q^\top(C^u\!-\!I)Q$ 把每用户求解从 $O(n)$ 降到 $O(\text{\#interactions})$。**评估必须用排序指标**（Recall@K/NDCG），不能用 RMSE。
> **English**: **Implicit vs explicit**: industry is almost all implicit. **Negative sampling** is key — with no true negatives, sample from the unobserved as negatives (BPR). **BPR loss** = $-\log\sigma(\hat x_{ui}-\hat x_{uj})$, i observed, j sampled. **iALS** trick: confidence $c_{ui}=1+\alpha r_{ui}$, and $Q^\top C^u Q = Q^\top Q + Q^\top(C^u\!-\!I)Q$ cuts each user solve from $O(n)$ to $O(\#\text{interactions})$. **Evaluation must use ranking metrics** (Recall@K/NDCG), never RMSE.


In [ ]:

# ============================================================
# 把 MovieLens 转成隐式反馈 / Turn MovieLens into implicit feedback
# 中文：真实隐式数据(Last.fm 播放次数)不在本地，我们用标准做法：把"评分>=4"当作一次"正反馈/喜欢"，
#       其余（评分<4 或 未评分）全部视为"未观测"。这样 100k 评分变成一堆 (user, item) 正交互。
# English: Real implicit data (Last.fm play counts) isn't local; standard practice: treat rating>=4 as
#          one positive interaction ("liked"); everything else (rating<4 or unrated) is "unobserved".
# ============================================================
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(0)
R_DIR=os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
rat=pd.read_csv(os.path.join(R_DIR,"u.data"),sep="\t",names=["user","item","rating","ts"])
rs=rat.sort_values("ts"); tr=[];te=[]
for _,g in rs.groupby("user"):
    c=int(len(g)*0.8); tr.append(g.iloc[:c]); te.append(g.iloc[c:])
train=pd.concat(tr); test=pd.concat(te)
uids=np.sort(rat["user"].unique()); iids=np.sort(rat["item"].unique())
u2x={u:x for x,u in enumerate(uids)}; i2x={i:x for x,i in enumerate(iids)}
m,n=len(uids),len(iids)

def positives(df):                                   # 评分>=4 视为正交互 / rating>=4 = positive
    d=df[df["rating"]>=4]
    return list(zip(d["user"].map(u2x).values, d["item"].map(i2x).values))
trp=positives(train); tep=positives(test)

seen=[set() for _ in range(m)]                       # 每个用户训练里交互过的物品 / train positives
for u,i in trp: seen[u].add(i)
test_pos=[[] for _ in range(m)]                      # 测试集正样本（排除训练已见）/ held-out positives
for u,i in tep:
    if i not in seen[u]: test_pos[u].append(i)
pop=np.zeros(n)                                       # 物品流行度基线 / popularity baseline
for u,i in trp: pop[i]+=1
print(f"训练正交互 train positives: {len(trp)}, 测试正样本 test positives: {sum(len(t) for t in test_pos)}")
print(f"用户 m={m}, 物品 n={n}, 平均每用户训练交互 / avg train pos per user: {len(trp)/m:.1f}")


**中文**：先实现 **BPR**。它的世界观是"排序"：对用户 $u$、一个他交互过的正物品 $i$、一个随机采样的未交互负物品 $j$，我们希望模型给 $i$ 的分高于 $j$。把"$i$ 排在 $j$ 前面"的概率建模为 $\sigma(\hat x_{ui}-\hat x_{uj})$，最大化它的对数似然。
**English**: First, **BPR**. Its worldview is ranking: for user $u$, a positive item $i$ they interacted with, and a randomly sampled negative $j$ they did not, we want the model to score $i$ above $j$. Model "$i$ ranks above $j$" as $\sigma(\hat x_{ui}-\hat x_{uj})$ and maximize its log-likelihood.

$$\mathcal{L} = \sum_{(u,i,j)} -\log\sigma(\hat x_{ui}-\hat x_{uj}) + \lambda\|\Theta\|^2,\qquad \hat x_{ui}=b_i+\mathbf{p}_u\!\cdot\!\mathbf{q}_i$$

**中文**：对每个采样三元组求梯度，记 $s=\sigma(\hat x_{uj}-\hat x_{ui})$（即正样本"还不够高"的程度），则 $\mathbf{p}_u$ 的梯度方向是 $s(\mathbf{q}_i-\mathbf{q}_j)$——把用户向量推向正物品、拉离负物品。这就是负采样在起作用。
**English**: For each sampled triple, with $s=\sigma(\hat x_{uj}-\hat x_{ui})$ (how much the positive is "not yet high enough"), the gradient for $\mathbf{p}_u$ points along $s(\mathbf{q}_i-\mathbf{q}_j)$ — pushing the user vector toward the positive and away from the negative. That is negative sampling at work.


In [ ]:

# ============================================================
# BPR：对 (u, 正i, 采样负j) 做 SGD / BPR-SGD over (u, pos i, sampled neg j)
# ============================================================
def train_bpr(k=32, lr=0.05, reg=0.01, epochs=15):
    rng=np.random.default_rng(0)
    P=rng.normal(0,0.1,(m,k)); Q=rng.normal(0,0.1,(n,k)); bi=np.zeros(n)
    ui=np.array([u for u,_ in trp]); ii=np.array([i for _,i in trp]); Npos=len(trp)
    for ep in range(epochs):
        for t in rng.permutation(Npos):              # 每轮遍历所有正样本一次 / one pass over positives
            u,i=ui[t],ii[t]
            j=rng.integers(n)                         # 采样一个负物品 / sample a negative item
            while j in seen[u]: j=rng.integers(n)     # 确保 j 是用户未交互的 / ensure unobserved
            x=bi[i]-bi[j]+P[u]@(Q[i]-Q[j])           # 正负分差 / score gap
            s=1.0/(1.0+np.exp(x))                      # s=sigma(-x)=梯度系数 / gradient coefficient
            Pu=P[u].copy()
            P[u]+=lr*(s*(Q[i]-Q[j])-reg*P[u])         # 用户向量：推向正、离开负 / toward pos, away neg
            Q[i]+=lr*(s*Pu - reg*Q[i])                # 正物品向量 / positive item
            Q[j]+=lr*(-s*Pu - reg*Q[j])               # 负物品向量 / negative item
            bi[i]+=lr*(s-reg*bi[i]); bi[j]+=lr*(-s-reg*bi[j])
    return P,Q,bi

t0=time.time(); Pb,Qb,bib = train_bpr(); print(f"BPR 训练耗时 / train time: {time.time()-t0:.1f}s")

# Top-N 排序评估工具 / top-N ranking evaluation helper
def eval_topN(score_of_user, N=10):
    Ps=Rs=0.0; cnt=0
    for u in range(m):
        if not test_pos[u]: continue
        sc=score_of_user(u).copy()
        for i in seen[u]: sc[i]=-1e9                  # 屏蔽训练已见 / mask training items
        top=np.argpartition(sc,-N)[-N:]               # 取分数最高的 N 个 / top-N (unordered ok)
        rec=set(top.tolist()); rel=set(test_pos[u])
        h=len(rec&rel); Ps+=h/N; Rs+=h/len(rel); cnt+=1
    return Ps/cnt, Rs/cnt, cnt

bpr_p,bpr_r,nu = eval_topN(lambda u: bib + Qb@Pb[u])
pop_p,pop_r,_  = eval_topN(lambda u: pop.copy())
print(f"评估用户 / users: {nu}")
print(f"BPR        P@10={bpr_p:.4f}  R@10={bpr_r:.4f}")
print(f"Popularity P@10={pop_p:.4f}  R@10={pop_r:.4f}")


**中文**：再实现 **iALS（隐式 ALS / 加权矩阵分解）**。思路不同于 BPR：它**保留所有格子**，把观测到的设为偏好 $p_{ui}=1$、未观测设 $0$，但给每个格子一个**置信度** $c_{ui}=1+\alpha\,r_{ui}$（这里 $r$ 是交互次数，本数据是 1）。损失是**置信度加权的平方误差**：
**English**: Next, **iALS (implicit ALS / weighted MF)**. Unlike BPR it **keeps all cells**, setting preference $p_{ui}=1$ for observed and $0$ otherwise, but assigns each cell a **confidence** $c_{ui}=1+\alpha\,r_{ui}$ ($r$ = interaction count, here 1). The loss is **confidence-weighted squared error**:

$$\min_{P,Q}\sum_{u,i} c_{ui}\,(p_{ui}-\mathbf{p}_u\!\cdot\!\mathbf{q}_i)^2 + \lambda(\|P\|^2+\|Q\|^2)$$

**中文**：关键技巧让它高效：固定 $Q$ 时每个用户的闭式解是 $\mathbf{p}_u=(Q^\top C^u Q+\lambda I)^{-1}Q^\top C^u \mathbf{p}(u)$。直接算 $Q^\top C^u Q$ 要遍历全部 $n$ 个物品，但利用 $Q^\top C^u Q = Q^\top Q + \alpha\sum_{i\in\text{obs}} \mathbf{q}_i\mathbf{q}_i^\top$（$Q^\top Q$ 全用户共享、只算一次），就只需遍历该用户**观测到的**物品。这是 iALS 能上工业规模的根本原因。
**English**: A key trick makes it efficient: with $Q$ fixed, each user's closed form is $\mathbf{p}_u=(Q^\top C^u Q+\lambda I)^{-1}Q^\top C^u \mathbf{p}(u)$. Computing $Q^\top C^u Q$ over all $n$ items is costly, but $Q^\top C^u Q = Q^\top Q + \alpha\sum_{i\in\text{obs}} \mathbf{q}_i\mathbf{q}_i^\top$ ($Q^\top Q$ shared across users, computed once) means we only loop over the user's **observed** items — why iALS scales to industry.


In [ ]:

# ============================================================
# iALS：置信度加权 ALS（Hu-Koren-Volinsky 2008）/ confidence-weighted ALS
# ============================================================
def train_ials(k=32, reg=0.1, alpha=40, epochs=12):
    rng=np.random.default_rng(1)
    P=rng.normal(0,0.01,(m,k)); Q=rng.normal(0,0.01,(n,k))
    items_of=[[] for _ in range(m)]; users_of=[[] for _ in range(n)]
    for u,i in trp: items_of[u].append(i); users_of[i].append(u)
    Ik=np.eye(k)*reg
    for ep in range(epochs):
        YtY=Q.T@Q                                     # 全用户共享项，每轮算一次 / shared, once per epoch
        for u in range(m):
            its=items_of[u]
            if not its: continue
            Qi=Q[its]                                 # 该用户观测物品的隐向量 / observed item vectors
            A=YtY + alpha*(Qi.T@Qi) + Ik              # Q^T C^u Q + λI（利用共享项的技巧）/ the trick
            b=(1+alpha)*Qi.sum(0)                      # Q^T C^u p(u): p=1 处权重 c=1+α / rhs
            P[u]=np.linalg.solve(A,b)                  # 闭式解 / closed form
        XtX=P.T@P
        for i in range(n):
            us=users_of[i]
            if not us: continue
            Pu=P[us]
            A=XtX + alpha*(Pu.T@Pu) + Ik
            b=(1+alpha)*Pu.sum(0)
            Q[i]=np.linalg.solve(A,b)
    return P,Q

t0=time.time(); Pi,Qi = train_ials(); print(f"iALS 训练耗时 / train time: {time.time()-t0:.1f}s")
ials_p,ials_r,_ = eval_topN(lambda u: Qi@Pi[u])
print(f"\n{'方法/method':<22}{'Precision@10':>14}{'Recall@10':>12}")
print(f"{'BPR':<22}{bpr_p:>14.4f}{bpr_r:>12.4f}")
print(f"{'iALS (weighted MF)':<22}{ials_p:>14.4f}{ials_r:>12.4f}")
print(f"{'Popularity':<22}{pop_p:>14.4f}{pop_r:>12.4f}")


**中文**：下面把三件事可视化：① BPR 随训练轮数的 Recall@10 上升曲线（看它如何从随机逐渐学会排序）；② iALS 的置信度系数 $\alpha$ 对效果的影响（$\alpha$ 太小= 几乎不区分正负，太大=过度自信于噪声正样本）；③ 三种方法 + 热门基线在 Recall@K 上随 K 的对比。
**English**: We visualize three things: ① BPR's Recall@10 rising over epochs (how it learns to rank from random); ② the effect of iALS's confidence factor $\alpha$ (too small = barely distinguishes pos/neg, too large = overconfident in noisy positives); ③ Recall@K vs K for all three methods plus the popularity baseline.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(15,4.3))

# ① BPR 学习曲线：每若干轮评一次 Recall@10 / BPR learning curve
def bpr_curve(k=32,lr=0.05,reg=0.01,epochs=15):
    rng=np.random.default_rng(0)
    P=rng.normal(0,0.1,(m,k));Q=rng.normal(0,0.1,(n,k));bi=np.zeros(n)
    ui=np.array([u for u,_ in trp]);ii=np.array([i for _,i in trp]);Npos=len(trp); curve=[]
    for ep in range(epochs):
        for t in rng.permutation(Npos):
            u,i=ui[t],ii[t]; j=rng.integers(n)
            while j in seen[u]: j=rng.integers(n)
            x=bi[i]-bi[j]+P[u]@(Q[i]-Q[j]); s=1.0/(1.0+np.exp(x)); Pu=P[u].copy()
            P[u]+=lr*(s*(Q[i]-Q[j])-reg*P[u]); Q[i]+=lr*(s*Pu-reg*Q[i]); Q[j]+=lr*(-s*Pu-reg*Q[j])
            bi[i]+=lr*(s-reg*bi[i]); bi[j]+=lr*(-s-reg*bi[j])
        _,r,_=eval_topN(lambda u: bi+Q@P[u]); curve.append(r)
    return curve
curve=bpr_curve()
ax[0].plot(range(1,len(curve)+1),curve,"o-",color="#4C72B0")
ax[0].axhline(pop_r,ls="--",color="#C44E52",label=f"popularity R@10={pop_r:.3f}")
ax[0].set_title("BPR 学习曲线 / BPR learning curve"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("Recall@10"); ax[0].legend()

# ② iALS 的 alpha 影响 / effect of confidence alpha
alphas=[1,5,15,40,100,300]
ials_recall=[]
for a in alphas:
    Pa,Qa=train_ials(alpha=a,epochs=10); _,r,_=eval_topN(lambda u: Qa@Pa[u]); ials_recall.append(r)
ax[1].plot(alphas,ials_recall,"s-",color="#55A868")
ax[1].set_xscale("log"); ax[1].set_title("iALS 置信度 α 的影响 / effect of α"); ax[1].set_xlabel("alpha (log)"); ax[1].set_ylabel("Recall@10")
besta=alphas[int(np.argmax(ials_recall))]; ax[1].axvline(besta,ls="--",color="red",label=f"best α={besta}"); ax[1].legend()

# ③ Recall@K 对比 / Recall@K curves
Ks=[5,10,20,50]
def rec_at(scorefn,Ks):
    return [eval_topN(scorefn,N=K)[1] for K in Ks]
ax[2].plot(Ks,rec_at(lambda u: bib+Qb@Pb[u],Ks),"o-",label="BPR",color="#4C72B0")
ax[2].plot(Ks,rec_at(lambda u: Qi@Pi[u],Ks),"s-",label="iALS",color="#55A868")
ax[2].plot(Ks,rec_at(lambda u: pop.copy(),Ks),"^--",label="Popularity",color="#C44E52")
ax[2].set_title("Recall@K 对比 / Recall@K"); ax[2].set_xlabel("K"); ax[2].set_ylabel("Recall@K"); ax[2].legend()
plt.tight_layout(); plt.savefig("/tmp/rec04_viz.png",dpi=80); plt.show()
print("iALS 最佳 α / best alpha:", besta, " Recall@10=",round(max(ials_recall),4))


**中文**：诚实总结结果：
**English**: Honest summary of results:

**中文**：
1. **BPR 和 iALS 都明显打败热门基线**——这正是隐式 CF 的价值：在"只有点击、没有评分"的真实数据上做出个性化排序。注意对比 15.2：那里按预测评分排序的 CF 反而输给热门，而这里**专门为排序而训练**的模型赢了，再次印证"优化目标要和评估目标一致"。
2. **BPR 偏精度、iALS 偏召回**（具体哪个高取决于数据/超参）——BPR 直接优化 pairwise 顺序，头部更准；iALS 把所有物品都纳入加权回归，覆盖更广。两者没有绝对优劣。
3. **iALS 训练快得多**（闭式解 + 全用户共享 $Q^\top Q$），且天然可并行、易分布式——这是它在工业界（尤其大规模隐式数据）长盛不衰的原因；BPR 则胜在灵活（损失里可加任意特征）。
4. **$\alpha$ 有甜点**：太小则正负样本几乎同权、学不出偏好；太大则对噪声正样本过度自信。

**English**:
1. **Both BPR and iALS clearly beat popularity** — the value of implicit CF: personalized ranking on real "clicks-only, no ratings" data. Contrast 15.2: there, CF ranked by predicted rating *lost* to popularity, while here models **trained specifically for ranking** win — again confirming "align the training objective with the evaluation objective."
2. **BPR leans precision, iALS leans recall** (which wins depends on data/hyperparams) — BPR optimizes pairwise order directly (sharper at the head); iALS folds all items into weighted regression (broader coverage). Neither is universally better.
3. **iALS trains much faster** (closed form + shared $Q^\top Q$) and is naturally parallel/distributable — why it endures in industry (especially large-scale implicit data); BPR wins on flexibility (arbitrary features in the loss).
4. **$\alpha$ has a sweet spot**: too small and positives/negatives weigh almost equally (no preference learned); too large and the model is overconfident in noisy positives.

> 💼 **实战视角 / Practical angle**
> **中文**：现代召回（retrieval）阶段大量用 BPR/iALS 思想训练 embedding：**负采样**（in-batch / 全局随机 / hard negative）几乎是所有召回模型的标配。记住面试金句——*"隐式反馈没有真负样本，建模的核心就是怎么造负样本、怎么给正样本置信度。"* 下一节起进入**特征交叉**时代（FM/Wide&Deep/DeepFM），从"只有 ID"走向"ID+丰富特征"。
> **English**: Modern retrieval stages heavily use BPR/iALS-style embedding training: **negative sampling** (in-batch / global random / hard negatives) is standard in nearly all retrieval models. Interview one-liner — *"implicit feedback has no true negatives; modeling is all about how you fabricate negatives and assign confidence to positives."* From the next section we enter the era of **feature crossing** (FM/Wide&Deep/DeepFM), moving from "IDs only" to "IDs + rich features."

---
### 小结 / Summary
- **中文**：隐式反馈 = 只有正样本+未观测，不能当回归做；要么 pairwise 排序(BPR)，要么置信度加权 MF(iALS)。
- **English**: Implicit feedback = positives + unobserved only; not a regression. Either pairwise ranking (BPR) or confidence-weighted MF (iALS).
- **中文**：BPR 靠负采样，灵活；iALS 闭式解+共享 $Q^\top Q$，快且可扩展；二者都用排序指标评估。
- **English**: BPR relies on negative sampling (flexible); iALS uses closed form + shared $Q^\top Q$ (fast, scalable); both evaluated by ranking metrics.
- **中文**：训练目标必须对齐评估目标——为排序而训练，才能在排序上赢。
- **English**: Align training with evaluation — train for ranking to win at ranking.
